In [1]:
import pandas as pd
import numpy as np
import warnings
from utils import loeo_grid_search, select_final_params_loeo, save_final_params

# Cấu hình hiển thị
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
warnings.filterwarnings('ignore')

# --- HÀM HỖ TRỢ TÍNH SCORE VÀ LẤY TOP 10 ---
def calculate_top_10(raw_df, alpha_hot):
    """
    Hàm này thực hiện giống select_final_params_loeo nhưng trả về DataFrame Top 10
    thay vì chỉ in ra màn hình.
    """
    # 1. Xác định các cột tham số (W_...)
    param_cols = [c for c in raw_df.columns if c.startswith("W_")]
    
    # 2. Gom nhóm (Groupby) các tham số giống nhau từ các fold
    # Tính trung vị (median) của accuracy và tau
    agg_df = raw_df.groupby(param_cols)[["held_acc", "tau_hot_train"]].median().reset_index()
    
    # 3. Đổi tên cho khớp với main.py
    agg_df = agg_df.rename(columns={
        "held_acc": "median_held_acc", 
        "tau_hot_train": "median_tau_hot"
    })
    
    # 4. TÍNH SCORE (Đây là bước main.py làm ngầm)
    agg_df["score"] = agg_df["median_held_acc"] + alpha_hot * agg_df["median_tau_hot"]
    
    # 5. Sắp xếp giảm dần theo Score và lấy 10 dòng đầu
    return agg_df.sort_values(by="score", ascending=False).head(10)

print("✅ Đã chuẩn bị xong hàm tính toán!")

✅ Đã chuẩn bị xong hàm tính toán!


# Config

In [2]:
# CẤU HÌNH (Copy y hệt main.py)
CSV_PATH = "dataset/elephants_kruger_2007_transition.csv"
R_HOT = 1.5
SEED = 42
ALPHA_HOT = 0.2
EPS_SCORE = 1e-4

# Load Data
print(f"📂 Đang đọc dữ liệu từ: {CSV_PATH}...")
df_raw = pd.read_csv(CSV_PATH)
print(f"✅ Dữ liệu sẵn sàng: {len(df_raw)} dòng, {df_raw['episode'].nunique()} episodes.")

# Định nghĩa Grid (Dùng chung cho cả 2 chế độ để công bằng)
W_vel_grid = [0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
W_temp_grid = [0, 0.25, 0.5, 0.75, 1.0]
W_smooth_grid = [0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
W_entropy_grid = [0, 0.25, 0.5, 0.75, 1.0]

print("✅ Đã thiết lập Grid Search Space.")

📂 Đang đọc dữ liệu từ: dataset/elephants_kruger_2007_transition.csv...
✅ Dữ liệu sẵn sàng: 283674 dòng, 14 episodes.
✅ Đã thiết lập Grid Search Space.


In [ ]:
print("=" * 60)
print("🚀 BẮT ĐẦU: NORMAL ECO MODE (Giả định Chuẩn)")
print("=" * 60)

# 1. Chạy Grid Search (Trả về kết quả thô từng fold)
res_normal = loeo_grid_search(
    df_raw=df_raw,
    W_vel_grid=W_vel_grid,
    W_temp_grid=W_temp_grid,
    W_smooth_grid=W_smooth_grid,
    W_entropy_grid=W_entropy_grid,
    R_HOT=R_HOT,
    seed=SEED,
    verbose=True,      
    grid_size=50,
    hot_quantile=0.8,
    use_bad_eco=False
)

# 2. Tính toán Top 10 (Sử dụng hàm ở Cell 1)
print("\n🏆 BẢNG XẾP HẠNG NORMAL MODE (Top 10):")
top_10_normal = calculate_top_10(res_normal, ALPHA_HOT)
display(top_10_normal) # Hiển thị đẹp trong Notebook

# 3. Lưu kết quả tốt nhất (Dùng hàm gốc để đảm bảo format json chuẩn)
best_normal = select_final_params_loeo(res_normal, verbose=False, alpha_hot=ALPHA_HOT, eps_score=EPS_SCORE)
save_final_params(best_normal, output_path="dataset_preprocessed/final_params_normal.json")

🚀 BẮT ĐẦU: NORMAL ECO MODE (Giả định Chuẩn)
Pre-computing features...
Computing astronomical day/night...
--- LOEO fold: held-out = AM105 ---
--- LOEO fold: held-out = AM107 ---
--- LOEO fold: held-out = AM108 ---
--- LOEO fold: held-out = AM110 ---
--- LOEO fold: held-out = AM239 ---
--- LOEO fold: held-out = AM253 ---
--- LOEO fold: held-out = AM254 ---
--- LOEO fold: held-out = AM255 ---
--- LOEO fold: held-out = AM306 ---
--- LOEO fold: held-out = AM307 ---
--- LOEO fold: held-out = AM308 ---
--- LOEO fold: held-out = AM91 ---
--- LOEO fold: held-out = AM93 ---
--- LOEO fold: held-out = AM99 ---

🏆 BẢNG XẾP HẠNG NORMAL MODE (Top 10):


,W_vel,W_temp,W_smooth,W_entropy,median_held_acc,median_tau_hot,score
594,0.7500,0.2500,1.5000,1.0000,0.9615,0.8077,1.1231
592,0.7500,0.2500,1.5000,0.5000,0.9615,0.8077,1.1231
557,0.7500,0.0000,1.5000,0.5000,0.9615,0.8077,1.1231
593,0.7500,0.2500,1.5000,0.7500,0.9615,0.8077,1.1231
559,0.7500,0.0000,1.5000,1.0000,0.9615,0.7949,1.1205
558,0.7500,0.0000,1.5000,0.7500,0.9615,0.7949,1.1205
549,0.7500,0.0000,1.0000,1.0000,0.9615,0.7949,1.1205
584,0.7500,0.2500,1.0000,1.0000,0.9615,0.7949,1.1205
591,0.7500,0.2500,1.5000,0.2500,0.9615,0.7949,1.1205
553,0.7500,0.0000,1.2500,0.7500,0.9615,0.7949,1.1205


Saved: dataset_preprocessed/final_params_normal.json


In [ ]:
print("=" * 60)
print("🧪 BẮT ĐẦU: BAD ECO MODE (Kiểm chứng Phản biện)")
print("=" * 60)

# 1. Chạy Grid Search (Với chế độ Bad Eco)
res_bad = loeo_grid_search(
    df_raw=df_raw,
    W_vel_grid=W_vel_grid,
    W_temp_grid=W_temp_grid,
    W_smooth_grid=W_smooth_grid,
    W_entropy_grid=W_entropy_grid,
    R_HOT=R_HOT,
    seed=SEED,
    verbose=True,
    grid_size=50,
    hot_quantile=0.8,
    use_bad_eco=True  # <--- TRUE: Chạy chế độ Bad Eco
)

# 2. Tính toán Top 10
print("\n💀 BẢNG XẾP HẠNG BAD MODE (Top 10):")
top_10_bad = calculate_top_10(res_bad, ALPHA_HOT)
display(top_10_bad)

# 3. Lưu kết quả
best_bad = select_final_params_loeo(res_bad, verbose=False, alpha_hot=ALPHA_HOT, eps_score=EPS_SCORE)
save_final_params(best_bad, output_path="dataset_preprocessed/final_params_bad.json")
print (best_bad)

🧪 BẮT ĐẦU: BAD ECO MODE (Kiểm chứng Phản biện)
Pre-computing features...
Computing astronomical day/night...
--- LOEO fold: held-out = AM105 ---
--- LOEO fold: held-out = AM107 ---
--- LOEO fold: held-out = AM108 ---
--- LOEO fold: held-out = AM110 ---
--- LOEO fold: held-out = AM239 ---
--- LOEO fold: held-out = AM253 ---
--- LOEO fold: held-out = AM254 ---
--- LOEO fold: held-out = AM255 ---
--- LOEO fold: held-out = AM306 ---
--- LOEO fold: held-out = AM307 ---
--- LOEO fold: held-out = AM308 ---
--- LOEO fold: held-out = AM91 ---
--- LOEO fold: held-out = AM93 ---
--- LOEO fold: held-out = AM99 ---

💀 BẢNG XẾP HẠNG BAD MODE (Top 10):


,W_vel,W_temp,W_smooth,W_entropy,median_held_acc,median_tau_hot,score
185,0.2500,0.0000,0.5000,0.0000,0.9231,0.8590,1.0949
181,0.2500,0.0000,0.2500,0.2500,0.9231,0.8590,1.0949
180,0.2500,0.0000,0.2500,0.0000,0.9231,0.8590,1.0949
1,0.0000,0.0000,0.0000,0.2500,0.9231,0.8462,1.0923
2,0.0000,0.0000,0.0000,0.5000,0.9231,0.8462,1.0923
190,0.2500,0.0000,0.7500,0.0000,0.9231,0.8462,1.0923
183,0.2500,0.0000,0.2500,0.7500,0.9231,0.8462,1.0923
179,0.2500,0.0000,0.0000,1.0000,0.9231,0.8462,1.0923
6,0.0000,0.0000,0.2500,0.2500,0.9231,0.8462,1.0923
182,0.2500,0.0000,0.2500,0.5000,0.9231,0.8462,1.0923


Saved: dataset_preprocessed/final_params_bad.json


In [6]:
# So sánh kết quả tốt nhất của 2 bên
print("📊 SO SÁNH KẾT QUẢ CUỐI CÙNG:")
compare_df = pd.DataFrame([
    best_normal,
    best_bad
], index=["Normal Mode", "Bad Mode"])

# Chỉ hiện các cột quan trọng
cols = ["W_vel", "W_temp", "W_smooth", "W_entropy", "median_held_acc", "score"]
display(compare_df[cols])

📊 SO SÁNH KẾT QUẢ CUỐI CÙNG:


,W_vel,W_temp,W_smooth,W_entropy,median_held_acc,score
Normal Mode,0.7500,0.0000,1.5000,0.5000,0.9615,1.1231
Bad Mode,0.2500,0.0000,0.2500,0.0000,0.9231,1.0949
